# Herschrijven — trainingen naar de nieuwe stijl

Draai de pijplijn stap voor stap, met de mens als poort op de plek waar dat telt.

**Secties 1, 2, 4 en 10 doen géén API-calls**, en in sectie 3 alleen de goedkope classificatie
van de vrije-tekst-aantekeningen (Haiku). Je leest in, joint, normaliseert de besluiten en
inspecteert wat het model straks precies te zien krijgt — vóórdat de dure schrijfcalls lopen.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env` naast dit notebook.
3. Twee inputbestanden: het **scoresheet** (scorer-velden + de handmatig ingevulde kolommen
   `actie_besluit`, `kern_reviewer`, `modus_reviewer` en `guidance_reviewer`)
   en het **bronsheet** (`id` / `name` / `herschreven` / `content`).
4. `vervolgtraining.json` naast dit notebook: de catalogus van 779 trainingen
   (`product_id` / `titel` / `summary`) waaruit de Vervolgstappen worden gekozen.
5. `vervolgtrainingen_tree.json`: dezelfde trainingen, ingedeeld naar
   domein > subdomein > onderwerp. Die indeling bepaalt mee welke vervolgtrainingen op de
   shortlist komen en hoe ze gegroepeerd worden. Ontbreekt het bestand, dan valt de selectie
   terug op alleen keyword-overlap.

**Twee assen bepalen wat er met een training gebeurt.** De *modus* (`overnemen` / `stijl` /
`format` / `volledig`) zegt hoeveel van de bestaande tekst mag veranderen; de goedgekeurde
*actualiseringen* staan daar los van en worden op elk niveau doorgevoerd. Sectie 2 laat de
verdeling zien, sectie 3b vult het voorstel in.

## 1. Config

Paden en knoppen. `importlib.reload` zorgt dat je edits in de `.py`-bestanden meteen meekomen.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib
import besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings
for m in (besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings):
    importlib.reload(m)
bes, uit, rw = besluiten, rewrite_output, rewrite_trainings

# TWEE scoresheets, en het onderscheid is belangrijk. `SCORED` is de ruwe invoer: alleen de
# scorer-velden plus wat de reviewer met de hand invult (`actie_besluit`, `kern_reviewer`).
# Sectie 3b leest dat bestand, bepaalt de modus en de Modules-NB, en schrijft het resultaat
# naar `SCORED_MET_MODUS`. Alles daarna -- inspecteren, herschrijven, batch -- leest
# `SCORED_MET_MODUS`. Doe je dat niet, dan draait de pijplijn op de defaults: elke training
# wordt `volledig` herschreven en krijgt de stabiele NB, zonder dat er iets misgaat wat je
# ziet. `_load_scored` waarschuwt daarom naar stderr als de kolommen ontbreken.
SCORED           = "/Users/hugovandenbelt/Downloads/PRIO_TRAININGEN_SCOREN.xlsx"
SCORED_MET_MODUS = "scoresheet_met_modus.xlsx"
SOURCE           = "/Users/hugovandenbelt/Downloads/Nieuwe lijst incl. herschreven en dagen.xlsx"
BESLUITEN        = "besluiten.xlsx"
OUT_DIR          = "herschreven"

# START telt over de WACHTRIJ, niet over het scoresheet. De wachtrij is het scoresheet in
# sheetvolgorde MINUS wat op modus `overnemen` staat en MINUS wat al in `herschreven.xlsx`
# staat. Staan er 10 van de 15 trainingen al in, dan is START=3 de vierde van de vijf die
# resteren -- niet sheetrij 3. De preview-cel in sectie 6 zet die twee nummeringen naast
# elkaar; draai hem voordat je de batch start.
START = 0      # alleen sectie 6: 0-based positie in de wachtrij om mee te beginnen
N     = None      # alleen sectie 6: aantal trainingen vanaf START (None = tot het einde)
IDS   = []     # alleen sectie 6: vul je hier training_id's in, dan wint dat van START/N.
               # Een id-selectie verschuift niet mee als de wachtrij korter wordt.

INSPECT = 2808    # alleen sectie 4/5/7/8: het ene training_id dat je onder de loep neemt

## 2. Inlezen, joinen en de populatie bekijken

Controleer eerst dat beide sheets goed binnenkomen en dat elke gescoorde training een bron heeft.

De uitsplitsing laat zien wat er automatisch herschreven wordt. Let op: een structurele
actualiteitsbreuk of `menselijke_input_nodig` stuurt een training **alleen** naar de mens als de
reviewer nog geen besluit heeft ingevuld. Heeft hij dat wel, dan is de menselijke poort al
gepasseerd en gaat de training gewoon mee.

In [22]:
import pandas as pd

scored = rw._load_scored(SCORED_MET_MODUS)
src_by_id, cols = rw.load_source(SOURCE)
print("scoresheet:", scored.shape, "| bronkolommen:", cols)

ontbreekt = [t for t in scored["training_id"] if t not in src_by_id]
print("zonder bron:", ontbreekt or "geen")

# de reviewer-poort: staat er een ingevuld besluit tegenover de acties van deze training?
besloten = {tid for tid, lijst in bes.load_besluiten(BESLUITEN).items()
            if any(x.besluit_ruw.strip() for x in lijst)}

# AS 1 -- de modus. Dit leest SCORED_MET_MODUS, dus dit is de verdeling die de batch straks
# ook draait. Staat hier alles op `volledig`, dan is dat het signaal dat sectie 3b nog moet
# lopen: zonder `modus_voorstel` valt de modus terug op de oude `herschreven`-kolom.
modus = scored.apply(
    lambda r: rw.build_briefing({k: r[k] for k in scored.columns}, {}, "").modus, axis=1)

overnemen     = modus == "overnemen"
onbruikbaar   = scored["verdict"] == "onbruikbaar"
beslissing    = ((scored["actualiteit_type"] == "structureel")
                 | scored["menselijke_input_nodig"].astype(bool))
wacht_op_mens = beslissing & ~scored["training_id"].isin(besloten)
auto          = ~(overnemen | onbruikbaar | wacht_op_mens)

print("\nAS 1 — herschrijfniveau:")
for m in rw.MODI:
    print(f"  {m:10} {int((modus == m).sum()):3d}")

print(f"""
doorgezet zonder herschrijven (overnemen) {int(overnemen.sum()):3d}
verdict onbruikbaar -> human-queue        {int(onbruikbaar.sum()):3d}
beslissing nodig, nog geen besluit        {int(wacht_op_mens.sum()):3d}
  (beslissing nodig, besluit ingevuld)    {int((beslissing & ~wacht_op_mens & ~overnemen).sum()):3d}
------------------------------------------------
gaat de auto-herschrijving in             {int(auto.sum()):3d}""")

# AS 2 -- de actualiseringen. Die lopen op ELK niveau mee, ook bij `overnemen`; daar krijgen
# ze een gerichte call die alleen de geraakte kopjes aanraakt.
met_acties = {tid for tid, lijst in bes.load_besluiten(BESLUITEN).items()
              if bes.splits(lijst)[0]}
n_overnemen_met_acties = int(scored[overnemen]["training_id"].isin(met_acties).sum())
print(f"\nAS 2 — trainingen met >=1 goedgekeurde actualisering: {len(met_acties)}"
      f"\n  waarvan op modus 'overnemen' (gerichte call, rest blijft letterlijk staan): "
      f"{n_overnemen_met_acties}")

scoresheet: (15, 38) | bronkolommen: {'id': 'id', 'name': 'name', 'content': 'content', 'days': None}
zonder bron: geen

AS 1 — herschrijfniveau:
  overnemen    0
  stijl        0
  format       8
  volledig     7

doorgezet zonder herschrijven (overnemen)   0
verdict onbruikbaar -> human-queue          0
beslissing nodig, nog geen besluit          0
  (beslissing nodig, besluit ingevuld)      2
------------------------------------------------
gaat de auto-herschrijving in              15

AS 2 — trainingen met >=1 goedgekeurde actualisering: 63
  waarvan op modus 'overnemen' (gerichte call, rest blijft letterlijk staan): 0


## 3. Besluiten normaliseren — de menselijke poort

`actie_besluit` heeft een vaste *structuur* (`<nr> <vrije tekst>`) maar vrije *tekst*.
Python splitst de structuur, een klein model classificeert de aantekening als
**doen / niet / mits**, en het resultaat landt in `besluiten.xlsx`.

Draai eerst de structuurcontrole (geen API). Daarna genereer je het sheet en kijk je
**alleen de regels met `bron=llm`** na. Corrigeer wat niet klopt, zet `bron` op `handmatig`
en draai de cel opnieuw — handmatige labels worden nooit overschreven.

**In dezelfde ronde: kijk de `kern` na.** Die kolom staat in het scoresheet, met een lege
kolom `kern_reviewer` ernaast. De kern legt het **niveau** van de training vast — er is
geen apart niveau-veld — en stuurt daarmee elk kopje dat de schrijver produceert. Klopt
hij, laat `kern_reviewer` leeg. Klopt hij niet, plak hem over en pas hem aan.

Waar je op let: beschrijft de kern wat de deelnemer met het onderwerp *doet*, in de
werkwoorden van de bron? Zegt de bron "maak je kennis met" en staat er in de kern "leert
toepassen", dan is dat de correctie. En sluit de kern af met één zin over wat de training
expliciet níét doet?

Een door jou bijgestelde kern is **leidend** over de brontekst; een kern die alleen van de
scorer komt verliest juist van de brontekst en levert hooguit een melding in `notities` op.

In [3]:
fouten = bes.check_alignment(SCORED)
assert not fouten, "los eerst de uitlijnfouten op"

51 rijen, 47 met genummerde acties, 0 uitlijnfouten


In [4]:
besluiten_df = bes.write_besluiten_sheet(SCORED, BESLUITEN, verbose=False)

na_te_kijken = besluiten_df[besluiten_df["bron"] == "llm"].sort_values("confidence")
print(f"{len(besluiten_df)} besluiten; {len(na_te_kijken)} door het model geclassificeerd\n")
pd.set_option("display.max_colwidth", 70)
na_te_kijken[["training_id", "nr", "besluit_ruw", "besluit", "confidence"]]

199 besluiten; 18 door het model geclassificeerd



,training_id,nr,besluit_ruw,besluit,confidence
0,5,1,"PHP versie niet benoemen, wel relavante taalfeatures toevoegen",mits,high
41,85,1,geen specifieke versies benoemen,mits,high
38,82,2,nuanceer,mits,high
36,47,3,voeg toestandsdiagram toe als programmaonderdeel,doen,high
34,47,1,geen specifieke tools benoemen,mits,high
31,46,2,geen versienummers gebruiken,mits,high
30,46,1,beide voor zover browser variant nog relevant,mits,high
22,27,3,in inleiding is dat prima,mits,high
20,27,1,nee dat is advanced,niet,high
17,12,1,stukje over certificeringsverwijzing volledig verwijderen,mits,high


## 3b. De modus bepalen — zelfde ronde, zelfde reviewer

Hoeveel moet er aan deze training gebeuren? Vier niveaus, elk niveau mag alles wat het
niveau eronder mag:

| Modus | Wat er mag veranderen |
| --- | --- |
| `overnemen` | Niets, behalve de titel en de vervolgtitels |
| `stijl` | De formulering, naar de **actuele** schrijfregels |
| `format` | + de structuur en de ontbrekende kopjes |
| `volledig` | + de opbouw, vanaf nul uit de brontekst |

**Python kan dit niet alleen.** `rewrite_checks.py` vangt openingszinnen, lengtes, aantallen
en verboden woorden — daarmee valt te *bewijzen dat een tekst niet voldoet, nooit dat hij wél
voldoet*. Of een zin het stijlregister volgt ziet code niet. Vandaar dezelfde drietrap als bij
de besluiten: `scan_vorm()` legt een deterministische ondergrens (en stelt dus nooit
`overnemen` voor), een Haiku-call leest de actuele schrijfspec naast de bestaande tekst, en jij
beslist in `modus_reviewer`.

Kijk vooral de regels na **waar het voorstel afwijkt van de ondergrens** — daar heeft het model
iets gezien wat de checks niet zagen, of het omgekeerde. Laat `modus_reviewer` leeg als je het
met het voorstel eens bent.

De goedgekeurde actualiseringen uit sectie 3 staan hier **los** van: die worden op elk niveau
doorgevoerd, ook bij `overnemen`. Een goedgekeurde actie is een lokale toevoeging en hoort de
rest van de training niet mee te slepen.

### De tweede uitkomst van deze cel: de Modules-NB

Dezelfde call levert `modules_nb_voorstel` op. Dat gaat **niet over de tekst maar over het
onderwerp**, en al helemaal niet over de actualiseringen hierboven: het bepaalt welke van twee
vaste zinnen onder kopje Modules komt te staan.

| Variant | De zin | Wanneer |
| --- | --- | --- |
| `stabiel` (default) | "…bel ons gerust als je de inhoud op jouw praktijksituatie aangepast wilt zien" | Verreweg de meeste trainingen |
| `actueel` | "…afhankelijk van snelle ontwikkelingen kan de werkelijke inhoud hiervan afwijken" | Alleen als het vakgebied zó snel beweegt dat de programmabeschrijving binnen een jaar achterloopt |

In de terminal herken je de tweede aan `[Modules-NB: voorbehoud-zin]`, met de motivering
eronder. Die voorbehoud-zin hoort de uitzondering te zijn — hij suggereert anders dat we zelf
niet weten wat we geven. Ben je het er niet mee eens, zet dan `stabiel` in
`modules_nb_reviewer`; die kolom wint altijd van het voorstel.

**Een training die hier `stabiel` krijgt kan nog steeds actualiseringen nodig hebben.** Dat zijn
twee losse dingen: de NB is een vaste zin in het document, de actualiseringen komen uit de
besluitenronde van sectie 3.

De cel weigert te draaien als een `training_id` niet in de bronsheet voorkomt. Dat is bijna
altijd een typefout in de id-kolom — een id met een punt erin (`2.347`) wordt als decimaal
gelezen en joint nergens meer mee.

**Let op waar je vanaf hier invult.** Deze cel schrijft `scoresheet_met_modus.xlsx`, en alles
daarna leest dat bestand — niet meer het ruwe scoresheet. `modus_reviewer`, `guidance_reviewer`
en `modules_nb_reviewer` vul je dus daarin in. `kern_reviewer` staat in het ruwe sheet en moet
vóór deze cel ingevuld zijn; draai je 3b later opnieuw, dan komt hij alsnog mee.

In [ ]:
# Leest SCORED (ruw) en schrijft SCORED_MET_MODUS; beide paden staan in sectie 1.
# met_llm=False -> alleen de deterministische ondergrens, geen API-call en geen key nodig.
# Bruikbaar als kalibratie; als voorstel niet, want de ondergrens stelt nooit `overnemen` voor.
voorstel = rw.modus_voorstellen(SCORED, SOURCE, SCORED_MET_MODUS, met_llm=True)

afwijkend = voorstel[voorstel["modus_voorstel"] != voorstel["modus_ondergrens"]]
print(f"\n{len(afwijkend)} rijen waar het model afwijkt van de ondergrens — die eerst nalezen:")
afwijkend[["training_id", "titel", "modus_ondergrens", "modus_voorstel", "modus_reden"]]

## 4. De briefing inspecteren — nog steeds zonder API-call

Dit is wat het model straks letterlijk krijgt. Controleer drie dingen:

- de **goedgekeurde** acties staan er, mét hun voorwaarde;
- de **afgewezen** acties staan onder NIET DOEN;
- `actualiteit_specifiek` en `actualiteit_samenvatting` staan er **niet** in — dat is
  onderbouwing van de scorer, geen besluit.

In [15]:
b = rw.build_briefing_for_id(SCORED_MET_MODUS, SOURCE, INSPECT, besluiten_path=BESLUITEN)
briefing = rw.build_writer_user(b)

print(f"{b.titel} | persona {b.persona} | {b.dagen} dagen | modus {b.modus} | "
      f"{len(b.goedgekeurd)} goedgekeurd, {len(b.afgewezen)} afgewezen\n")

# In stijl/format VERVANGT de bestaande tekst per kopje de brontekst -- zelfde content, maar
# per veld en compleet (`build_source_text` slaat setup/follow_up/summary_edudex/certification
# over). Knip het materiaalblok eraf zodat je de briefing zelf leest, niet de training.
grens = next((briefing.index(k) for k in ("OPDRACHT —", "BRONTEKST —", "HUIDIGE VERSIE —")
              if k in briefing), len(briefing))
print(briefing[:grens])

print(f"\n{'='*70}\nmateriaalblok ({'huidige versie per kopje' if b.behoudt_tekst else 'brontekst'})"
      f"\n{'='*70}\n{briefing[grens:][:1200]}")

rij = scored[scored.training_id == INSPECT].iloc[0]
for veld in ("actualiteit_specifiek", "actualiteit_samenvatting"):
    lek = str(rij[veld])[:60] in briefing
    print(f"\n{veld:28} lekt naar de schrijver: {lek}")

Training AWS Monitoring met CloudWatch | persona A | 2 dagen | modus volledig | 4 goedgekeurd, 0 afgewezen

Titel: Training AWS Monitoring met CloudWatch
Persona: A
Aantal dagen: 2
Verdict scorer: rijk

KERN (lezing van de scorer) — hierin staat het NIVEAU van de training; schrijf nooit boven dat
niveau, ook niet als een kopje om meer tekst vraagt:
De training gaat over het monitoren van het AWS platform met de nadruk op CloudWatch: metrics verzamelen, dashboards en custom metrics maken, logs inzichtelijk maken en alarms instellen, aangevuld met gerelateerde diensten als CloudTrail, GuardDuty, VPC Flow Logs en Systems Manager voor het loggen van activiteit en toegang. Het niveau is toepassingsgericht: de bron spreekt van "doornemen aan de hand van praktijkcases", modules als dashboards gebruiken, custom metrics maken, CloudWatch Agent toepassen, alarms instellen en het zelf opstellen van een monitoring-strategie in de afsluitende praktijkopdracht. Het zwaartepunt ligt op CloudWatch als

## 5. Eén training herschrijven

Vanaf hier lopen er API-calls. Schrijver → code-check → judge → route.

Het resultaat gaat meteen naar `herschreven/trainingen/`: `<id>.json` (lossless) en
`<id>.md` — precies het document dat je onder de cel leest, zodat je het later terugvindt.
`herschreven.xlsx` blijft hier ongemoeid; dat sheet vult de batch in sectie 6.

In [ ]:
catalog = rw.load_catalog()
boom    = rw.load_tree(catalog)   # vakgebied-indeling; stuurt de shortlist en de groepen
client  = rw.make_client()

print(f"catalogus: {len(catalog)} trainingen | in de boom ingedeeld: {len(boom['paden'])}")

res = rw.rewrite_one(client, b, catalog, boom)
print(f"status: {res.status}  {res.reden}")
print("flags:", res.flags or "geen")
print("toegepaste acties:", len(res.toegepaste_acties))

# meteen wegschrijven; de bron-content gaat mee zodat `days` overeind blijft in de JSON
bron_rij = src_by_id.get(b.training_id)
paden = rw.bewaar_training(
    OUT_DIR, res, rw.parse_content(bron_rij[cols["content"]]) if bron_rij is not None else {})
print("opgeslagen:", ", ".join(p for p in paden.values() if p), "\n")

print(uit.render_markdown(res.document, res.titel) if res.document else "(geen document)")

## 6. Batch draaien

`append=True` + `skip_existing=True`: een afgebroken run hervat zonder opnieuw te betalen.
Draai je dezelfde selectie nog eens, dan levert dat 0 nieuwe rijen op.

**Kijk eerst welke trainingen je raakt.** De volgorde is de rijvolgorde van het scoresheet —
er wordt nergens gesorteerd — maar `START` en `N` snijden pas ná twee filters: trainingen op
modus `overnemen` gaan hun eigen lus in, en alles wat al in `herschreven.xlsx` staat valt weg.
Wat overblijft is de **wachtrij**, en dáárover telt `START`. Sheetrij 3 en wachtrijpositie 3
zijn dus verschillende trainingen zodra er één rij is weggefilterd.

De cel hieronder maakt beide nummeringen zichtbaar en doet geen enkele API-call. Wil je niet
tellen, vul dan `IDS = [2808, 3036]` in sectie 1 in: die selectie verschuift niet mee als de
wachtrij korter wordt.

In [23]:
# Wat gaat de batch hieronder doen? Geen API-calls; `rewrite_file` bouwt zijn selectie met
# exact deze functie, dus dit is geen benadering maar dezelfde wachtrij.
# `alles=True` toont ook de rijen die al in herschreven.xlsx staan.
wachtrij = rw.toon_wachtrij(SCORED_MET_MODUS, OUT_DIR,
                            start=START, limit=N, alleen_ids=IDS or None)

wachtrij — 4 van 15 sheetrijen, in sheetvolgorde

  sheet  wachtrij      id  titel                                           modus    
      4         0    2808  Training AWS Monitoring met CloudWatch          volledig   (buiten START/N)
      5         1    3036  Training Change Management voor DAMA-DMBOK      volledig   (buiten START/N)
      6         2    2529  Training Cybersecurity voor developers          format     (buiten START/N)
     14  ->     3    2725  Training SIEM: Security Information and Event   volledig   << draait
  ... plus 11 rijen die al in herschreven/herschreven.xlsx staan (alles=True toont ze)

selectie: 1 te herschrijven (START=3, N=1)


In [ ]:
review = rw.rewrite_file(SCORED_MET_MODUS, SOURCE, OUT_DIR, besluiten_path=BESLUITEN,
                         start=START, limit=N, alleen_ids=IDS or None)

review[["training_id", "titel", "modus", "status", "reden", "thin", "n_flags", "spec_versie"]]

## 7. De output naast de bron leggen

De gegenereerde `content` heeft dezelfde sleutels als de bron, dus je kunt veld voor veld
vergelijken. Let op `days` en `certification` (ongewijzigd) en op de kop 3 in `intro`.

In [7]:
import json
from score_trainings import parse_content

with open(f"{OUT_DIR}/trainingen/{INSPECT}.json", encoding="utf-8") as f:
    resultaat = json.load(f)

nieuw = resultaat["content"]
oud   = parse_content(src_by_id[INSPECT][cols["content"]])
print("sleutels gelijk aan de bron:", set(nieuw) == set(oud))

for kopje in sjabloon.KOPJES:
    v = nieuw.get(kopje.cms, "")
    print(f"\n{'='*70}\n{kopje.kop}  ({kopje.cms})\n{'='*70}")
    print(v if isinstance(v, str) else repr(v))

sleutels gelijk aan de bron: True

Overzicht  (summary)
Wil je zelfstandig webapplicaties bouwen met PHP en MySQL? Tijdens deze opleiding leer je programmeren van functioneel naar objectgeoriënteerd, met security als vaste rode draad. Je werkt met Composer, PSR-standaarden en moderne taalfeatures, en bouwt een eigen webapplicatie zoals een webwinkel. Na afloop schrijf je onderhoudbare code, zet je een veilige database op en koppel je die aan je applicatie.

Inleiding  (intro)
<p>PHP draait achter een groot deel van het web, van maatwerkapplicaties tot bekende platformen. Tijdens deze opleiding bouw je in vijf dagen een stevige basis op als PHP-ontwikkelaar. Je begint met functioneel programmeren: variabelen, arrays, lussen, functies en het afhandelen van fouten. Daarna stap je over naar objectgeoriënteerd werken, waarbij je zowel externe als eigen classes inzet en kennismaakt met design patterns.</p>
<p>Databases vormen het tweede fundament. Je zet zelf een MySQL-database op, bevraagt 

## 8. Eén kopje bijsturen

Is er één kopje mis, dan hoef je de hele training niet opnieuw te genereren. Zet `KOPJE` op het
veld dat je wilt hergenereren en geef optioneel een `COMMENT` mee met wat er anders moet.
Zonder comment is het een gewone retry.

De schrijver krijgt de volledige huidige training als context, zodat het nieuwe kopje aansluit
op wat er al staat. De per-training-artefacten (`<id>.json` én `<id>.md`) en de rij in
`herschreven.xlsx` worden bijgewerkt.

In [ ]:
KOPJE   = "modules"    # kies uit rw.HERGENEREERBAAR
COMMENT = ""           # bv. "module 2 en 4 overlappen; voeg ze samen en voeg een module over X toe"

print("hergenereerbaar:", ", ".join(rw.HERGENEREERBAAR), "\n")

with open(f"{OUT_DIR}/trainingen/{INSPECT}.json", encoding="utf-8") as f:
    voor = json.load(f)
oud = rw._writer_out_uit_json(voor).get(KOPJE)

res = rw.hergenereer_kopje_op_schijf(SCORED_MET_MODUS, SOURCE, INSPECT, KOPJE, COMMENT,
                                     besluiten_path=BESLUITEN, out_dir=OUT_DIR)

print(f"\n{'='*70}\nOUD\n{'='*70}")
print(json.dumps(oud, ensure_ascii=False, indent=2)[:2000])
print(f"\n{'='*70}\nNIEUW\n{'='*70}")
print(json.dumps(res.writer_out.get(KOPJE), ensure_ascii=False, indent=2)[:2000])

## 10. Goud-corpus: kalibreren op de 78 bestaande trainingen

De trainingen die al in de nieuwe stijl staan (`herschreven=1`) worden **niet** herschreven:
ze gaan het sheet in op modus `overnemen` (status `overgenomen`). Als corpus zijn ze een
meetlat, geen voorbeeldmateriaal.

`checks_over_goud()` draait de code-check over alle 78 en telt per regel hoe vaak die omvalt.
Dat werkt twee kanten op:

- **valt een regel bij meer dan de helft om**, dan is de regel verdacht en niet de training.
  Zo viel het oude harde lengtevenster om, en is de lengte nu een richtlijn met een vangrail
  (`lengtes_over_goud()` meet de verdeling waarop die banden zijn gekozen);
- **de trainingen die álles halen** waren ooit de few-shot. Dat kan niet meer: sinds Templatev2
  en de modules-checks haalt er nul élke harde regel. De few-shot komt nu uit onze eigen
  output; zie sectie 11.

Let daarbij op `bullets_aantal`: dat valt vaak om over 78 trainingen. Lees dat als een vraag
aan de spec, niet als een oordeel over het goud, precies waar de eerste regel hierboven voor is.

Verander je een check, draai deze cel dan opnieuw.

In [ ]:
rw.export_goud_corpus(SOURCE, OUT_DIR)
print()
meting = rw.checks_over_goud()

# En dezelfde meting over de few-shot zelf. Die MOET schoon zijn: een voorbeeld dat een
# regel laat vallen, demonstreert precies de vorm die het hoort te weren.
print(f"\n{'='*70}\nfew-shot ({rw.GOUD_V2_DIR})\n{'='*70}")
v2 = rw.checks_over_goud(rw.GOUD_V2_DIR)
schoon = {str(t) for t, _ in v2["schoon"]}
vervuild = [t for t in rw.actieve_goud_voorbeelden() if t not in schoon]
if vervuild:
    print(f"\nLET OP: {vervuild} haalt niet meer alle harde checks. Draai sectie 11 opnieuw "
          f"of herstel de training.")
else:
    print(f"\nalle {len(rw.actieve_goud_voorbeelden())} few-shot-voorbeelden halen alle harde checks")

## 11. Eigen output tot goud promoveren

De few-shot hoort te bestaan uit trainingen die mét de huidige spec zijn geschreven en die
alle checks halen. `promoveer_naar_goud()` doet dat in één stap: hij draait de checks over
`herschreven/trainingen/`, kopieert wat slaagt naar `herschreven/goud_v2/` en legt de selectie
vast in `selectie.json`. `rw.GOUD_VOORBEELDEN` leest dat manifest bij import, dus er blijft
geen lijst met id's in `rewrite_trainings.py` achter die je met de hand moet bijwerken.

Drie dingen om te weten:

- **de checks gaan over de rijke vorm** (`writer_out` + de groepen uit het document), niet over
  de CMS-HTML zoals bij `checks_over_goud`. Daardoor zien ze ook `aanpak_invulling`, de
  catalogustitels en de groep-intro's;
- **de content wordt opnieuw gerenderd** uit het document, zodat een voorbeeld nooit verouderde
  boilerplate demonstreert;
- **`vervang=True` maakt de goudmap gelijk aan de nieuwe selectie.** De vier gerepareerde
  `v2_*`-voorbeelden zijn altijd terug te bouwen met `python bouw_goud_v2.py`.

De selectie is bewust **vast** en niet per training wisselend: de hele system-prefix gaat als
één gecachet blok mee, dus een prefix die per training verschilt maakt de prompt-cache
waardeloos. Er gaan er `rw.GOUD_N` mee, en welke dat waren staat per training in
`<id>.json` onder `goud_voorbeelden`.

Kijk bij het promoveren naar de profielregels: loopt het aantal dagen of het vakgebied te veel
gelijk, dan leert de schrijver één vorm in plaats van een stijl.

In [ ]:
# eerst kijken, dan pas schrijven
rw.promoveer_naar_goud(dry_run=True)

In [ ]:
uitslag = rw.promoveer_naar_goud()

import importlib
importlib.reload(rw)
print(f"\nfew-shot na herladen: {rw.actieve_goud_voorbeelden()}")